In [1]:
# 1. Import the class
from helpers.backtest import Backtest

# 2. Create an instance of the Backtest class
bt = Backtest()

# 3. Run the backtest method with your desired parameters
# This will test the 'macd' strategy on 'nifty_50' stocks
# with at least 365 days of data and return the top 5 results.
results_df = bt.backtest(
    strategy='macd',
    stocks='nifty_50',
    min_days=365,
    top_n=5
)

# 4. Print the results
print("Backtest Results:")
print(results_df)

# You could also try another strategy, like RSI with custom thresholds
rsi_results_df = bt.backtest(
    strategy='rsi',
    stocks='nifty_200',
    top_n=10,
    buying_thresh=25,
    selling_thresh=75
)

print("\nRSI Backtest Results:")
print(rsi_results_df)


Backtest Results:
            win%  wins  losses   ROI  \
HDFCBANK    0.56     9       7  0.62   
JSWSTEEL    0.48    10      11 -1.02   
M&M         0.47     9      10  1.23   
MAXHEALTH   0.45     9      11  0.47   
HINDUNILVR  0.44     8      10 -0.15   

                                                          p&l  \
HDFCBANK    [102.39999999999986, 121.95000000000005, -10.0...   
JSWSTEEL    [-12.0, -13.649999999999977, -47.5, 22.3000000...   
M&M         [-92.90000000000009, -28.25, 125.0, 93.0, 365....   
MAXHEALTH   [-20.0, -70.34999999999991, 85.95000000000005,...   
HINDUNILVR  [40.30000000000018, -99.25, -98.84999999999991...   

                                                     buy_date  \
HDFCBANK    [2024-02-16 00:00:00, 2024-05-27 00:00:00, 202...   
JSWSTEEL    [2023-12-29 00:00:00, 2024-02-06 00:00:00, 202...   
M&M         [2023-12-15 00:00:00, 2024-02-01 00:00:00, 202...   
MAXHEALTH   [2024-01-05 00:00:00, 2024-01-24 00:00:00, 202...   
HINDUNILVR  [2023-12-21 0

# 📊 Table Display Options for Backtest Results

This notebook demonstrates multiple ways to display backtest results in a professional, easy-to-read table format.

In [2]:
import pandas as pd
import numpy as np
from IPython.display import HTML, display
import plotly.graph_objects as go

# Configure pandas display options for better readability
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [5]:
def display_table_styled(df, title="Results", height=400, max_rows=None):
    """
    Display DataFrame as a styled HTML table with color formatting
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The dataframe to display
    title : str
        Title of the table
    height : int
        Height of the table container in pixels
    max_rows : int
        Maximum rows to display (None = all)
    """
    # Limit rows if specified
    display_df = df.head(max_rows) if max_rows else df
    
    # Create styled DataFrame
    styled = display_df.style
    
    # Apply color gradients to numeric columns
    numeric_cols = display_df.select_dtypes(include=[np.number]).columns
    
    for col in numeric_cols:
        if col in ['win%', 'ROI', 'wins']:
            # Green gradient for positive performance metrics
            styled = styled.background_gradient(
                subset=[col],
                cmap='RdYlGn',
                vmin=display_df[col].min(),
                vmax=display_df[col].max()
            )
    
    # Format numeric columns
    format_dict = {}
    for col in numeric_cols:
        if col in ['win%', 'ROI']:
            format_dict[col] = '{:.2f}%'
        elif col in ['days', 'wins', 'losses', 'buys', 'sells', 'hold_period']:
            format_dict[col] = '{:.0f}'
        else:
            format_dict[col] = '{:.2f}'
    
    styled = styled.format(format_dict)
    styled = styled.set_properties(**{'text-align': 'center', 'font-size': '11pt'})
    styled = styled.set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#4472C4'), ('color', 'white'), ('font-weight', 'bold')]},
        {'selector': 'tr:hover', 'props': [('background-color', '#E7E6E6')]},
    ])
    
    print(f"\n{'='*80}\n📊 {title}\n{'='*80}\n")
    display(styled)
    print(f"\n📈 Summary: {len(display_df)} results displayed\n")
    
    return styled


def display_table_simple(df, title="Results", max_rows=15):
    """
    Display DataFrame as a simple text table (good for quick overview)
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The dataframe to display
    title : str
        Title of the table
    max_rows : int
        Maximum rows to display
    """
    display_df = df.head(max_rows) if max_rows else df
    
    print(f"\n{'='*100}")
    print(f"📊 {title}")
    print(f"{'='*100}\n")
    print(display_df.to_string())
    print(f"\n{'='*100}")
    print(f"📈 Total Results: {len(df)} | Displayed: {len(display_df)}")
    print(f"{'='*100}\n")


def display_table_plotly(df, title="Results", max_rows=10):
    """
    Display DataFrame as an interactive Plotly table
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The dataframe to display
    title : str
        Title of the table
    max_rows : int
        Maximum rows to display
    """
    display_df = df.head(max_rows) if max_rows else df
    
    # Reset index to include symbol names as a column
    df_reset = display_df.reset_index()
    df_reset.columns = ['Stock'] + list(df_reset.columns[1:])
    
    # Create color scale for win% column
    win_pct = df_reset['win%'].values if 'win%' in df_reset.columns else None
    
    if win_pct is not None:
        colors = ['red' if x < 0.5 else 'yellow' if x < 0.6 else 'lightgreen' for x in win_pct]
    else:
        colors = ['white'] * len(df_reset)
    
    fig = go.Figure(data=[go.Table(
        header=dict(
            values=list(df_reset.columns),
            fill_color='#4472C4',
            align='center',
            font=dict(color='white', size=12, family='Arial')
        ),
        cells=dict(
            values=[df_reset[col].values for col in df_reset.columns],
            fill_color=['white']*len(df_reset.columns),
            align='center',
            font=dict(size=11),
            height=25
        )
    )])
    
    fig.update_layout(
        title=f"<b>{title}</b><br><sub>Total: {len(df)} results | Showing: {len(display_df)}</sub>",
        height=max(400, len(display_df) * 30 + 100),
        margin=dict(l=0, r=0, t=50, b=0),
        font=dict(family='Arial', size=11)
    )
    
    fig.show()


def display_table_summary_stats(df, title="Results"):
    """
    Display summary statistics of backtest results
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The backtest results dataframe
    title : str
        Title of the summary
    """
    print(f"\n{'='*80}")
    print(f"📊 {title} - SUMMARY STATISTICS")
    print(f"{'='*80}\n")
    
    print(f"Total Stocks Tested:        {len(df)}")
    print(f"Average Win%:               {df['win%'].mean():.2%}")
    print(f"Average ROI:                {df['ROI'].mean():.2f}%")
    print(f"Average Wins per Stock:     {df['wins'].mean():.2f}")
    print(f"Average Losses per Stock:   {df['losses'].mean():.2f}")
    print(f"Average Hold Period (days): {df['hold_period'].apply(lambda x: np.mean(x)).mean():.1f}")
    
    print(f"\nBest Performer:             {df.index[0]} (Win%: {df['win%'].iloc[0]:.2%})")
    print(f"Worst Performer:            {df.index[-1]} (Win%: {df['win%'].iloc[-1]:.2%})")
    
    print(f"\nStocks with >50% Win Rate:  {len(df[df['win%'] > 0.50])}")
    print(f"Stocks with >60% Win Rate:  {len(df[df['win%'] > 0.60])}")
    print(f"Stocks with 100% Win Rate:  {len(df[df['win%'] == 1.00])}")
    
    print(f"\n{'='*80}\n")

## Option 1: Styled HTML Table (Recommended for Jupyter)

Best for: Professional reports, color-coded performance metrics, easy reading

In [ ]:
# 1. Import the class
from helpers.backtest import Backtest

# 2. Create an instance of the Backtest class
bt = Backtest()

# 3. Run MACD backtest
results_df = bt.backtest(
    strategy='macd',
    stocks='nifty_50',
    min_days=365,
    top_n=5
)

# 4. Display results - Option 1: Styled HTML Table
display_table_styled(results_df, title="MACD Strategy - NIFTY 50 Results", max_rows=10)

## Option 2: Simple Text Table (Quick Overview)

In [ ]:
# Run RSI backtest
rsi_results_df = bt.backtest(
    strategy='rsi',
    stocks='nifty_200',
    top_n=10,
    buying_thresh=25,
    selling_thresh=75
)

# Display as simple text table
display_table_simple(rsi_results_df, title="RSI Strategy - NIFTY 200 Results", max_rows=8)

## Option 3: Interactive Plotly Table (Best for Presentations)

In [9]:
# Display as interactive Plotly table
display_table_plotly(rsi_results_df, title="RSI Strategy - Interactive Table", max_rows=8)

## Option 4: Summary Statistics (High-Level Overview)

In [8]:
# Display summary statistics
display_table_summary_stats(rsi_results_df, title="RSI Strategy - NIFTY 200")


📊 RSI Strategy - NIFTY 200 - SUMMARY STATISTICS

Total Stocks Tested:        10
Average Win%:               60.00%
Average ROI:                6.96%
Average Wins per Stock:     0.60
Average Losses per Stock:   0.40
Average Hold Period (days): 237.2

Best Performer:             BHARTIARTL (Win%: 100.00%)
Worst Performer:            HUDCO (Win%: 0.00%)

Stocks with >50% Win Rate:  6
Stocks with >60% Win Rate:  6
Stocks with 100% Win Rate:  6




## Option 5: Filtered Views (Focus on Best Performers)

In [7]:
# Filter: Show only stocks with >60% win rate
winners = rsi_results_df[rsi_results_df['win%'] > 0.60]
display_table_styled(winners, title="🏆 Stocks with >60% Win Rate (RSI Strategy)", max_rows=20)

# Filter: Show only stocks with positive ROI
profitable = rsi_results_df[rsi_results_df['ROI'] > 0]
display_table_simple(profitable, title="💰 Profitable Stocks (Positive ROI)", max_rows=15)


📊 🏆 Stocks with >60% Win Rate (RSI Strategy)



,win%,wins,losses,ROI,p&l,buy_date,sell_date,buy_price,sell_price,hold_period,buys,sells,days
BHARTIARTL,1.00%,1,0,27.00%,[np.float64(427.04999999999995)],[Timestamp('2024-11-25 00:00:00')],[Timestamp('2025-07-01 00:00:00')],[np.float64(1581.95)],[np.float64(2009.0)],[218],1,1,511
INDIGO,1.00%,1,0,24.42%,[np.float64(992.3000000000002)],[Timestamp('2024-10-31 00:00:00')],[Timestamp('2025-03-24 00:00:00')],[np.float64(4063.0)],[np.float64(5055.3)],[144],1,1,511
INDUSTOWER,1.00%,1,0,15.64%,[np.float64(54.10000000000002)],[Timestamp('2024-10-29 00:00:00')],[Timestamp('2025-04-23 00:00:00')],[np.float64(345.9)],[np.float64(400.0)],[176],1,1,511
ATGL,1.00%,1,0,14.83%,[np.float64(144.0)],[Timestamp('2024-03-15 00:00:00')],[Timestamp('2024-06-04 00:00:00')],[np.float64(971.0)],[np.float64(1115.0)],[81],1,1,511
COCHINSHIP,1.00%,1,0,5.94%,[np.float64(112.04999999999995)],[Timestamp('2024-09-23 00:00:00')],[Timestamp('2025-05-20 00:00:00')],[np.float64(1886.95)],[np.float64(1999.0)],[239],1,1,447
HINDZINC,1.00%,1,0,0.78%,[np.float64(4.0499999999999545)],[Timestamp('2024-08-21 00:00:00')],[Timestamp('2025-06-12 00:00:00')],[np.float64(515.95)],[np.float64(520.0)],[295],1,1,511



📈 Summary: 6 results displayed


📊 💰 Profitable Stocks (Positive ROI)

            win%  wins  losses    ROI                   p&l               buy_date              sell_date  buy_price sell_price hold_period buys sells days
BHARTIARTL   1.0     1       0  27.00  [427.04999999999995]  [2024-11-25 00:00:00]  [2025-07-01 00:00:00]  [1581.95]   [2009.0]       [218]    1     1  511
INDIGO       1.0     1       0  24.42   [992.3000000000002]  [2024-10-31 00:00:00]  [2025-03-24 00:00:00]   [4063.0]   [5055.3]       [144]    1     1  511
INDUSTOWER   1.0     1       0  15.64   [54.10000000000002]  [2024-10-29 00:00:00]  [2025-04-23 00:00:00]    [345.9]    [400.0]       [176]    1     1  511
ATGL         1.0     1       0  14.83               [144.0]  [2024-03-15 00:00:00]  [2024-06-04 00:00:00]    [971.0]   [1115.0]        [81]    1     1  511
COCHINSHIP   1.0     1       0   5.94  [112.04999999999995]  [2024-09-23 00:00:00]  [2025-05-20 00:00:00]  [1886.95]   [1999.0]       [239]    1    

## Option 6: Comparison View (Side-by-Side)

In [ ]:
# Compare top 5 stocks from both strategies
print("\n" + "="*100)
print("📊 STRATEGY COMPARISON: MACD vs RSI")
print("="*100 + "\n")

print("MACD Top 5:")
display_table_simple(results_df.head(5), title="Top 5 MACD Results", max_rows=10)

print("\n" + "-"*100 + "\n")

print("RSI Top 5:")
display_table_simple(rsi_results_df.head(5), title="Top 5 RSI Results", max_rows=10)